In [ ]:
import ee
from tgbs_rs.config.config import GEE_PROJECT

In [ ]:
# Authenticate and Initialize Earth Engine
ee.Authenticate()
ee.Initialize(project= GEE_PROJECT)

In [ ]:
from tgbs_rs.utils import build_default_sites_featurecollection, get_sites_geometry
from tgbs_rs.config.config import (
    BASELINE_START,
    CURRENT_START,
    CURRENT_END,
    CHIRPS_PRECIP_BAND,
    DRIVE_FOLDER,
)
from tgbs_rs.io import export_table_to_drive
from tgbs_rs.data.sensors.chirps.chirps_preprocessing import get_chirps_collection
from tgbs_rs.metrics.temporal import build_period_composites, collection_to_region_timeseries

In [5]:
# Build Site Feature Collection
sites_fc = build_default_sites_featurecollection()

# Get Sites Geometry
sites_geom = get_sites_geometry(sites_fc)

##### Build the full-window daily CHIRPS collection

In [6]:
chirps_full_daily = get_chirps_collection(
    aoi=sites_geom,
    start_date=BASELINE_START,
    end_date=CURRENT_END,
)

##### Build the current-window daily CHIRPS collection

In [7]:
chirps_current_daily = get_chirps_collection(
    aoi=sites_geom,
    start_date=CURRENT_START,
    end_date=CURRENT_END,
)

##### Build annual total rainfall composites for the full time window

In [10]:
chirps_annual_total = build_period_composites(
    collection=chirps_full_daily,
    bands=[CHIRPS_PRECIP_BAND],
    start_date=BASELINE_START,
    end_date=CURRENT_END,
    temporal_scale="annual",
    composite_stat="sum",
)

##### Build monthly total rainfall composites for the current time window

In [11]:
chirps_monthly_total = build_period_composites(
    collection=chirps_current_daily,
    bands=[CHIRPS_PRECIP_BAND],
    start_date=CURRENT_START,
    end_date=CURRENT_END,
    temporal_scale="monthly",
    composite_stat="sum",
)

##### Reduce each composite collection over the AOI region

In [12]:
chirps_annual_total_fc = collection_to_region_timeseries(
    collection=chirps_annual_total,
    region=sites_geom,
    bands=[CHIRPS_PRECIP_BAND],
    reducer=ee.Reducer.mean(),
    scale=5566,
    tile_scale=4,
)

In [13]:
chirps_monthly_total_fc = collection_to_region_timeseries(
    collection=chirps_monthly_total,
    region=sites_geom,
    bands=[CHIRPS_PRECIP_BAND],
    reducer=ee.Reducer.mean(),
    scale=5566,
    tile_scale=4,
)

##### Export rainfall tables to Drive

In [ ]:
export_table_to_drive(
    collection=chirps_annual_total_fc,
    description="tgbs_chirps_v3_rnl_annual_total_2014_2025",
    folder=DRIVE_FOLDER,
    fileNamePrefix="tgbs_chirps_v3_rnl_annual_total_2014_2025",
)

In [ ]:
export_table_to_drive(
    collection=chirps_monthly_total_fc,
    description="tgbs_chirps_v3_rnl_monthly_total_2018_2025",
    folder=DRIVE_FOLDER,
    fileNamePrefix="tgbs_chirps_v3_rnl_monthly_total_2018_2025",
)